# 01 · Run the full application from Colab
Starts the FastAPI backend (which also serves the React dashboard) with Google Drive as storage, and opens
a Cloudflare quick tunnel so you can use the dashboard from your browser.

The tunnel URL is public, so the API is protected with a random access token included in the printed link.
Keep the link private. Stop the last cell (or the runtime) to shut everything down.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Installs the project code into Drive the first time (upload dist/qadt_code.zip built with
# scripts/make_colab_bundle.sh), then reuses it in every later session.
import os, sys, zipfile
CODE_DIR = '/content/drive/MyDrive/Quantum_Agentic_Digital_Twin_code'
if not os.path.isdir(os.path.join(CODE_DIR, 'backend')):
    print(f'Project code not found in {CODE_DIR}. Choose qadt_code.zip to upload it:')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file uploaded. Build it locally with: scripts/make_colab_bundle.sh')
    os.makedirs(CODE_DIR, exist_ok=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall(CODE_DIR)
    print('Code installed in Drive at', CODE_DIR)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
%pip install -q -r requirements.txt
!apt-get -qq install -y unrar > /dev/null   # the IMS archive contains .rar files
print('Ready:', CODE_DIR)

In [ ]:
from backend.storage import setup_storage, prepare_datasets
setup_storage()
prepare_datasets()        # no-op when the datasets are already in Drive

### Optional: local LLM with Ollama
Without an LLM the agents use deterministic narratives. To enable one, run this cell (downloads ~2 GB). Alternatively set `OPENAI_API_KEY` and choose the OpenAI provider in System Settings.

In [ ]:
USE_OLLAMA = False
if USE_OLLAMA:
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    import subprocess, time
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    time.sleep(5)
    !ollama pull llama3.2:3b
    import os; os.environ['QADT_LLM_MODEL'] = 'llama3.2:3b'

### Start the API and the tunnel

In [ ]:
import os, re, secrets, subprocess, time, urllib.request
token = secrets.token_urlsafe(24)
env = {**os.environ, 'QADT_API_TOKEN': token, 'PYTHONPATH': CODE_DIR}
api = subprocess.Popen(['python', '-m', 'uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
                       cwd=CODE_DIR, env=env, stdout=open('/content/api.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=1); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open('/content/api.log').read()[-3000:])
print('API running.')

if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\nOpen the dashboard (keep this link private):\n', f'{url}/?token={token}')

### Logs

In [ ]:
print(open('/content/api.log').read()[-4000:])